# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook acts as a simulator generating synthetic FHIR Observation & Patient streams via PySpark memory streams, writing them into the local Bronze Iceberg table created in Day 1.

In [ ]:
import os
import json
import time
import uuid
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, udf
from pyspark.sql.types import StringType

# Set up local Spark session
spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("Spark Session initialized.")

In [ ]:
# Generate synthetic FHIR JSON payloads
def generate_fhir_payload(record_id):
    return json.dumps({
        "resourceType": "Patient",
        "id": f"pat-{record_id}",
        "active": True,
        "name": [{"family": "Doe", "given": [f"John{record_id}"]}],
        "gender": "male"
    })

generate_fhir_udf = udf(generate_fhir_payload, StringType())

# Use Rate source to simulate a stream of incoming events
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 2) \
    .load()

# Transform into the Bronze raw payload schema
bronze_stream_df = streaming_df \
    .withColumn("payload_id", udf(lambda: str(uuid.uuid4()), StringType())()) \
    .withColumn("source_system_id", lit("SYNTHETIC_FHIR_STREAM")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("raw_payload_json", generate_fhir_udf("value")) \
    .select("payload_id", "source_system_id", "ingestion_timestamp", "raw_payload_json")

# Write the stream to the local Iceberg Bronze table
query = bronze_stream_df.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .option("path", "local.ehdip_bronze.raw_payloads") \
    .option("checkpointLocation", "/tmp/ehdip_warehouse/checkpoints/synthetic_fhir_stream/") \
    .start()

print("Streaming started. Writing to local.ehdip_bronze.raw_payloads...")

In [ ]:
# Run stream for a short duration then stop
import time
time.sleep(15)
query.stop()
print("Streaming stopped.")

# Query the Bronze table to verify records were ingested
spark.sql("SELECT * FROM local.ehdip_bronze.raw_payloads ORDER BY ingestion_timestamp DESC LIMIT 5").show(truncate=False)